<div style="text-align: center; margin-top: 80px;">

# **Universidad Internacional de la Rioja (UNIR)**

## Escuela Superior de Ingeniería y Tecnología
## Máster en Computación Cuántica

## Computación Cuántica
# **Actividad 1. Exploración del espacio de estados de un cúbit **


### Actividad de la asignatura 
### presentada por: Rodrigo Hernández Sacristán
### Profesor: Andrés Navas Cáliz

#### Fecha: 1 de diciembre de 2025  

---

</div>

# Simulación Interactiva de un Cúbit en la Esfera de Bloch

Este notebook permite explorar el **espacio de estados de un cúbit** y visualizar cómo las diferentes **puertas cuánticas** afectan su estado en la **esfera de Bloch**.

Incluye:

- Visualización de un estado cuántico inicial.
- Aplicación de puertas: Pauli (I, X, Y, Z), Hadamard, S, T, V (√X) y sus adjuntas.
- Dos Bloch Spheres: a la izquierda el estado inicial, a la derecha el estado tras la puerta.


En primer lugar importemos los paquetes y librerías necesarias para la realización, además se imprime en pantalla las versiones de Python y Qiskit que han de usarse para el correcto uso de la simulación.

In [1]:
####################
#     IMPORTS      #
####################
!pip install matplotlib
import sys
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.quantum_info import Statevector, Pauli
from ipywidgets import interact, Dropdown

# Mostrar versión de Python

print("Versión de Python:", sys.version)

# Mostrar versión de Qiskit
import qiskit
print("Versión de Qiskit:", qiskit.__version__)


  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 28.5 MB/s  0:00:00 eta 0:00:01
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 36.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 39.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [matplotlib]7 [matplotlib]


ModuleNotFoundError: No module named 'matplotlib'

# Definición de la Bloch Sphere

Creamos la clase **BlochSphere** para dibujar una esfera, los ejes X, Y, Z y un vector de estado.


In [ ]:
class BlochSphere:
    def __init__(self, ax=None, figsize=(9,9)):
        if ax is None:
            self.fig = plt.figure(figsize=figsize)
            self.ax = self.fig.add_subplot(111, projection="3d")
        else:
            self.ax = ax
            self.fig = ax.figure
        
        try:
            self.ax.set_box_aspect([1,1,1])
        except Exception:
            pass
        
        self._setup_axes()

    def _setup_axes(self):
        """Oculta ticks, cuadrícula y spines para que la esfera quede limpia"""
        self.ax.set_xticks([])
        self.ax.set_yticks([])
        self.ax.set_zticks([])
        self.ax.grid(False)
        if hasattr(self.ax, "spines"):
            for spine in self.ax.spines.values():
                spine.set_visible(False)

    def draw_sphere(self):
        """
        Dibuja la esfera de Bloch rosa, los meridianos finos y los ejes con etiquetas.
        Los polos |0⟩ y |1⟩ se muestran fuera de la esfera.
        """
        # Crear malla esférica
        u = np.linspace(0, 2*np.pi, 60)
        v = np.linspace(0, np.pi, 60)
        x = np.outer(np.sin(u), np.sin(v))
        y = np.outer(np.cos(u), np.sin(v))
        z = np.outer(np.ones_like(u), np.cos(v))

        # Superficie rosa semitransparente
        self.ax.plot_surface(
            x, y, z,
            rstride=1, cstride=1,
            color="#F7C9C9", alpha=0.35, linewidth=0
        )

        # Meridianos finos
        for angle in np.linspace(0, 2*np.pi, 12):
            self.ax.plot(
                np.sin(v)*np.cos(angle),
                np.sin(v)*np.sin(angle),
                np.cos(v),
                color="gray", alpha=0.25
            )

        # Ejes que atraviesan la esfera
        max_range = 1
        self.ax.plot([max_range, -max_range], [0,0], [0,0], color="black", linewidth=1)  # Y
        self.ax.plot([0,0], [-max_range, max_range], [0,0], color="black", linewidth=1)  # X
        self.ax.plot([0,0], [0,0], [-max_range, max_range], color="black", linewidth=1)  # Z

        # Etiquetas de los ejes
        self.ax.text(max_range + 0.2, 0, 0, "Y", fontsize=14, color="black")
        self.ax.text(0,max_range + 0.6, 0, "X", fontsize=14, color="black")
        self.ax.text(0, 0, max_range + 0.2, "Z", fontsize=14, color="black")

        # Polos escritos fuera de la esfera
        self.ax.text(0, 0, max_range + 0.4, "|0⟩", fontsize=16)
        self.ax.text(0, 0, -max_range - 0.4, "|1⟩", fontsize=16)

        # Forzar aspecto 1:1:1
        try:
            self.ax.set_box_aspect([1,1,1])
        except Exception:
            self.ax.get_proj = lambda: np.dot(self.ax.get_proj(), np.diag([1,1,1,1]))

        # Limitar ejes para que la esfera sea perfectamente redonda
        self.ax.set_ylim([max_range, -max_range])
        self.ax.set_xlim([-max_range, max_range])
        self.ax.set_zlim([-max_range, max_range])

    def add_statevector(self, statevector, color="crimson"):
        """
        Añade un vector de estado a la esfera.
        Convención: X → fondo, Y → derecha, Z → arriba
        """
        sv = Statevector(statevector)
        bx = np.real(sv.expectation_value(Pauli("X")))
        by = np.real(sv.expectation_value(Pauli("Y")))
        bz = np.real(sv.expectation_value(Pauli("Z")))

        # Dibujar flecha desde el origen
        self.ax.quiver(
            0, 0, 0,
            by, bx, bz,   # convención y,x,z para que coincida con Bloch real
            color=color,
            linewidth=3,
            arrow_length_ratio=0.15
        )

    def show(self, title="Bloch Sphere"):
        """Muestra la figura final"""
        self.ax.set_title(title, fontsize=18, pad=20)
        plt.show()


Ahora vamos a utilizar dos formas diferentes para aplicar las puertas cuánticas.

***Primera forma***

## Definición de las puertas cuánticas como matrices
Aquí definimos las matrices de las puertas cuánticas de forma manual usando NumPy.



In [ ]:
I = np.array([[1,0],[0,1]])
X = np.array([[0,1],[1,0]])
Y = np.array([[0,-1j],[1j,0]])
Z = np.array([[1,0],[0,-1]])
H = 1/np.sqrt(2)*np.array([[1,1],[1,-1]])
S = np.array([[1,0],[0,1j]])
Sdg = np.array([[1,0],[0,-1j]])
T = np.array([[1,0],[0,np.exp(1j*np.pi/4)]])
Tdg = np.array([[1,0],[0,np.exp(-1j*np.pi/4)]])
V = 1/2*np.array([[1+1j,1-1j],[1-1j,1+1j]]) # √X
Vdg = np.linalg.inv(V)


## Función para aplicar puerta y dibujar Bloch Sphere
Se dibujan **dos esferas**: a la izquierda el estado inicial y a la derecha el estado tras aplicar la puerta cuántica seleccionada.


In [ ]:
def apply_gate_and_plot(statevector, gate_name):
    """Aplica la puerta seleccionada y dibuja el estado inicial y el estado final"""
    # Diccionario de puertas
    gates = {
        "Puerta Pauli I": I,
        "Puerta Pauli X": X,
        "Puerta Pauli Y": Y,
        "Puerta Pauli Z": Z,
        "Puerta Hadamard": H,
        "Puerta S": S,
        "Puerta adjunta de S": Sdg,
        "Puerta T": T,
        "Puerta adjunta de T": Tdg,
        "Puerta V (√X)": V,
        "Puerta adjunta de V ((√X)†)": Vdg
    }
    
    # Evolucionar el estado
    new_state = Statevector(gates[gate_name] @ statevector.data)
    
    # Crear figura con dos subplots
    fig = plt.figure(figsize=(16,8))
    
    # Estado inicial
    ax1 = fig.add_subplot(121, projection='3d')
    bloch_init = BlochSphere(ax=ax1)
    bloch_init.draw_sphere()
    bloch_init.add_statevector(statevector)
    ax1.set_title(f"Estado inicial", fontsize=16)
    
    # Estado tras aplicar puerta
    ax2 = fig.add_subplot(122, projection='3d')
    bloch_gate = BlochSphere(ax=ax2)
    bloch_gate.draw_sphere()
    bloch_gate.add_statevector(new_state)
    ax2.set_title(f"Estado tras aplicar {gate_name}", fontsize=16)
    
    plt.show()


## Estado inicial y menú interactivo
Escribe las coordenadas del vector estado en la base canónica entre corchetes en el argumento de la función Statevector, es decir, si queremos visualizar el vector estado $\ket{1}=0\ket{0}+1\ket{1}$, hemos de escribir Statevector([0,1]) y en la esfera de Bloch de la izquierda veremos este estado representado. Después seleccionamos la puerta que queramos aplicar y observamos cómo cambia el estado de partida del cúbit en la esfera de Bloch. En el ejemplo que demuestra el correcto funcionamiento estamos partiendo del estado $\ket{+}=\frac{1}{\sqrt(2)}\ket{0}+\frac{1}{\sqrt(2)}\ket{1}.$


In [ ]:
# Estado inicial
initial_state = Statevector([1/np.sqrt(2), 1/np.sqrt(2)])

# Crear menú interactivo
interact(
    apply_gate_and_plot,
    statevector=Dropdown(
        options=[initial_state], 
        value=initial_state, 
        description="Estado:"
    ),
    gate_name=Dropdown(
        options=[
            "Puerta Pauli I", "Puerta Pauli X", "Puerta Pauli Y", "Puerta Pauli Z",
            "Puerta Hadamard", "Puerta S", "Puerta adjunta de S",
            "Puerta T", "Puerta adjunta de T", "Puerta V (√X)", "Puerta adjunta de V ((√X)†)"
        ],
        value="Puerta Pauli I",
        description="Puerta:"
    )
)


NameError: name 'Statevector' is not defined

***Segunda forma***

## Función para crear un QuantumCircuit que incluya la puerta seleccionada
Según el nombre que escojamos en el menú, se construye un circuito de 1 cúbit
y se le aplica la puerta correspondiente.


In [ ]:
def apply_gate_and_plot(statevector, gate_name):

    qc = QuantumCircuit(1)

    if gate_name == "Puerta Pauli I":
        pass
    elif gate_name == "Puerta Pauli X":
        qc.x(0)
    elif gate_name == "Puerta Pauli Y":
        qc.y(0)
    elif gate_name == "Puerta Pauli Z":
        qc.z(0)
    elif gate_name == "Puerta Hadamard":
        qc.h(0)
    elif gate_name == "Puerta S":
        qc.s(0)
    elif gate_name == "Puerta adjunta de S":
        qc.sdg(0)
    elif gate_name == "Puerta T":
        qc.t(0)
    elif gate_name == "Puerta adjunta de T":
        qc.tdg(0)
    elif gate_name == "Puerta V (√X)":
        qc.sx(0)
    elif gate_name == "Puerta adjunta de V ((√X)†)":
        qc.sxdg(0)

    final_state = statevector.evolve(qc)

    fig = plt.figure(figsize=(12, 6))

    ax1 = fig.add_subplot(1, 2, 1, projection='3d')
    bloch_init = BlochSphere(ax=ax1)
    bloch_init.draw_sphere()
    bloch_init.add_statevector(statevector)
    ax1.set_title("Estado inicial")

    ax2 = fig.add_subplot(1, 2, 2, projection='3d')
    bloch_final = BlochSphere(ax=ax2)
    bloch_final.draw_sphere()
    bloch_final.add_statevector(final_state)
    ax2.set_title(f"Estado tras {gate_name}")

    plt.show()


## Estado inicial y menú interactivo
De la misma manera que antes, para probar la simulación hay que escribir las coordenadas del vector estado en la base canónica entre corchetes en el argumento de la función Statevector, es decir, si queremos visualizar el vector estado $\ket{1}=0\ket{0}+1\ket{1}$, hemos de escribir Statevector([0,1]) y en la esfera de Bloch de la izquierda veremos este estado representado. Después seleccionamos la puerta que queramos aplicar y observamos cómo cambia el estado de partida del cúbit en la esfera de Bloch. En el ejemplo que demuestra el correcto funcionamiento estamos partiendo del estado $\ket{+}=\frac{1}{\sqrt(2)}\ket{0}+\frac{1}{\sqrt(2)}\ket{1}.$



In [ ]:
initial_state = Statevector([1/np.sqrt(2), 1/np.sqrt(2)])

interact(
    apply_gate_and_plot,
    statevector=Dropdown(
        options=[initial_state],
        value=initial_state,
        description="Estado:"
    ),
    gate_name=Dropdown(
        options=[
            "Puerta Pauli I", 
            "Puerta Pauli X",
            "Puerta Pauli Y",
            "Puerta Pauli Z",
            "Puerta Hadamard",
            "Puerta S",
            "Puerta adjunta de S",
            "Puerta T",
            "Puerta adjunta de T",
            "Puerta V (√X)",
            "Puerta adjunta de V ((√X)†)"
        ],
        value="Puerta Pauli I",
        description="Puerta:"
    )
)


interactive(children=(Dropdown(description='Estado:', options=(Statevector([0.70710678+0.j, 0.70710678+0.j],
 …

<function __main__.apply_gate_and_plot(statevector, gate_name)>

Vemos que de las dos formas obtenemos los mismos resultados y podemos observar también que el comportamiento de las puertas cuánticas en la simulación es exactamente el mismo que se expone en la memoria. 